# Lecture 04: Neural language models and attention

CS40008.01 · Baojian Zhou · Fudan University · September 30, 2026

**How can a model select useful context instead of compressing everything into one state?**

Predict each result before running its cell. E01–E05 are **ungraded classroom practices**, in slide order, with the same expectations for all students. This notebook runs offline on CPU with the course environment (`uv sync`). It uses PyTorch, included through the existing environment, and no downloaded datasets, tokenizers, or model weights. All text, Q/K/V values, and measurements below are explicitly **toy examples**.

| Exercise | Time | Student response |
| --- | --- | --- |
| E01 | 4 min | Context/target pairs and tensor shapes |
| E02 | 6 min | Repair a detached loss, then fit a consistent tiny batch |
| E03 | 5 min | One attention row and its output vector |
| E04 | 4 min | Allowed positions and future-token effects |
| E05 | 6 min | Output, gradient, and causality checks |
| P01 | Optional | Dense attention score storage and FLOPs |
| P02 | Optional | Permuting paired keys and values |

The cells include runnable explanations after the prediction prompts. To experiment, edit a personal copy under `workspace/`. The course Notebook launcher preserves existing copies.

In [ ]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(7)
print('PyTorch:', torch.__version__, '| device: CPU')

## A longer context than Lecture 03

Our invented corpus has two sentences:

```
BOS red  key opens  EOS
BOS blue key closes EOS
```

Both prefixes end in `key`, but the next token differs. A bigram model uses the same distribution after `key` in both cases. With equal counts, its best distribution on these two targets assigns probability 1/2 to each and has mean loss $\log 2$ nats. A context of two tokens can distinguish them.

Vocabulary IDs are local to this notebook. `BOS` marks a sentence start and `EOS` a sentence end. We allocate one output row per vocabulary entry, including `BOS`, although `BOS` is never a training target. No window crosses a sentence boundary, and nothing is predicted after `EOS`.

In [ ]:
vocabulary = ['BOS', 'red', 'blue', 'key', 'opens', 'closes', 'EOS']
token_to_id = {token: index for index, token in enumerate(vocabulary)}
sentences = ['BOS red key opens EOS', 'BOS blue key closes EOS']
documents = [[token_to_id[token] for token in text.split()] for text in sentences]


def make_windows(documents, context_size):
    if context_size < 1:
        raise ValueError('context_size must be positive')
    contexts, targets = [], []
    for document in documents:
        for position in range(context_size, len(document)):
            contexts.append(document[position - context_size:position])
            targets.append(document[position])
    return (torch.tensor(contexts, dtype=torch.long).reshape(-1, context_size),
            torch.tensor(targets, dtype=torch.long))


C, D, H, V = 2, 4, 8, len(vocabulary)
train_contexts, train_targets = make_windows(documents, C)
for context, target in zip(train_contexts, train_targets):
    print([vocabulary[index] for index in context], '->', vocabulary[target])

## E01 · Context windows and shapes

**4 minutes.** For `BOS red key opens EOS`, enumerate the three length-2 contexts and their targets. Then take a batch of **two** contexts with $C=2$, embedding width $d=4$, hidden width $h=8$, and vocabulary size $|V|=7$. Predict the shapes after lookup, concatenation, hidden transformation, and vocabulary projection.

The network follows the NPLM example in the [Spring notebook, §3](https://github.com/baojian/llm-26/blob/main/lecture-04-neural-lms/lecture-04-neural-lms.ipynb). It uses plain `nn.Module` and a local vocabulary. We omit the optional direct input-to-output connection from [Bengio et al. (2003), §2](https://www.jmlr.org/papers/v3/bengio03a.html).

In [ ]:
class ContextLM(torch.nn.Module):
    def __init__(self, vocab_size, context_size, width, hidden_width):
        super().__init__()
        self.context_size = context_size
        self.embedding = torch.nn.Embedding(vocab_size, width)
        self.hidden = torch.nn.Linear(context_size * width, hidden_width)
        self.output = torch.nn.Linear(hidden_width, vocab_size)

    def forward(self, ids):
        if ids.ndim != 2 or ids.shape[1] != self.context_size:
            raise ValueError('ids must have shape (batch, context_size)')
        x = self.embedding(ids).flatten(start_dim=1)
        return self.output(torch.tanh(self.hidden(x)))


torch.manual_seed(7)
model = ContextLM(V, C, D, H)
e01_ids = train_contexts[:2]
e01_lookup = model.embedding(e01_ids)
e01_flat = e01_lookup.flatten(start_dim=1)
e01_hidden = torch.tanh(model.hidden(e01_flat))
e01_shapes = {
    'ids': tuple(e01_ids.shape), 'lookup': tuple(e01_lookup.shape),
    'concatenated': tuple(e01_flat.shape), 'hidden': tuple(e01_hidden.shape),
    'logits': tuple(model(e01_ids).shape),
}
print(e01_shapes)

**Check:** `[BOS, red] -> key`, `[red, key] -> opens`, `[key, opens] -> EOS`.
Shapes: `(2, 2)`, `(2, 2, 4)`, `(2, 8)`, `(2, 8)`, `(2, 7)`.
Concatenation preserves context order. The hidden nonlinearity lets the network learn interactions between positions. Without nonlinearities, consecutive affine layers collapse to one affine map.

### Stable next-token loss

Each context has **one** next-token target: logits have shape $(B,|V|)$ and targets $(B,)$. Supply raw logits to `F.cross_entropy`. Its stable form is $\log\sum_j e^{z_j}-z_y$.

Subtracting the largest logit before exponentiation avoids overflow in softmax. `logsumexp` computes the corresponding log-normalizer stably. Large logits can produce infinity in a direct `exp`; the next cell deliberately demonstrates that failure.

In [ ]:
large_logits = torch.tensor([[1000., 1001., 999.]], dtype=torch.float64)
target = torch.tensor([1])
naive_normalizer = large_logits.exp().sum(dim=-1)
stable_loss = (torch.logsumexp(large_logits, dim=-1)
               - large_logits.gather(-1, target[:, None]).squeeze(-1)).mean()
builtin_loss = F.cross_entropy(large_logits, target)
uniform_loss = F.cross_entropy(torch.zeros(2, V), train_targets[:2]).item()
print('naive normalizer:', naive_normalizer.tolist())
print('stable / PyTorch:', stable_loss.item(), builtin_loss.item())
print('uniform baseline:', uniform_loss, '| log(V):', math.log(V))

## E02 · Repair the training loop

**6 minutes.** A student writes:

```python
loss = F.cross_entropy(model(train_contexts), train_targets).detach()
loss.backward()
```

Explain why this fails, then repair it. Where should gradients be cleared? Predict what `optimizer.step()` changes. Run the corrected loop below and inspect the six predictions.

This is a deliberately consistent batch: no identical length-2 context has conflicting targets. The debugging goal is a final mean loss below **0.03 nats** after 200 SGD updates. Achieving this says nothing yet about new text.

In [ ]:
broken_model = ContextLM(V, C, D, H)
detached_loss = F.cross_entropy(broken_model(train_contexts), train_targets).detach()
try:
    detached_loss.backward()
except RuntimeError as error:
    detached_error = str(error)
    print('Expected failure:', detached_error)

In [ ]:
torch.manual_seed(7)
model = ContextLM(V, C, D, H)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
model.train()
e02_initial_embedding = model.embedding.weight.detach().clone()
with torch.no_grad():
    e02_losses = [F.cross_entropy(model(train_contexts), train_targets).item()]
e02_gradient_norms = []
for step in range(200):
    optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(model(train_contexts), train_targets)
    loss.backward()
    norm = torch.sqrt(sum(parameter.grad.square().sum() for parameter in model.parameters()))
    e02_gradient_norms.append(norm.item())
    optimizer.step()
    with torch.no_grad():
        e02_losses.append(F.cross_entropy(model(train_contexts), train_targets).item())
model.eval()
with torch.no_grad():
    e02_predictions = model(train_contexts).argmax(dim=-1)
print('loss before / after:', e02_losses[0], e02_losses[-1])
print('predictions:', [vocabulary[index] for index in e02_predictions])
print('correct:', int((e02_predictions == train_targets).sum()), '/', len(train_targets))

**Check:** removing `.detach()` preserves the path from the loss to the parameters. Clear gradients before `backward()`, then update after `backward()`. Detach only the scalar you record, or use `loss.item()`.

The predictions should be `key, opens, EOS, key, closes, EOS`. The losses and gradient norms should remain finite. Cross-entropy starts near the uniform baseline and ends below 0.03 nats with the specified initialization and learning rate.

This notebook intentionally has **no held-out evaluation**. For a generalization experiment, reserve separate documents before making overlapping windows, use the same tokenizer and scoring convention, and report held-out loss separately. `model.eval()` changes training-mode modules; `torch.no_grad()` disables gradient recording. Neither creates held-out data.

### Recurrence and selecting context

A feedforward LM keeps a fixed window. A recurrent LM updates a state, for example $h_t=\tanh(W_x x_t+W_h h_{t-1}+b)$. Gradients travel through repeated updates. LSTM gates control which cell-state information is retained, written, and exposed. They can help preserve information; they do not remove the sequential dependency.

Early encoder–decoder systems summarized the source in one fixed-length vector. [Bahdanau et al. (ICLR 2015), §§2–3](https://arxiv.org/abs/1409.0473) motivate a query-dependent weighted combination of source states. Their additive cross-attention is a historical bridge. We now implement **scaled dot-product self-attention**, whose Q/K/V come from the same input sequence.

For further study, the original [Spring slides, pp. 38–83](https://github.com/baojian/llm-26/blob/main/slides/lecture-04-slides/lecture-04-slides.pdf) and [Spring notebook, §§2 and 4](https://github.com/baojian/llm-26/blob/main/lecture-04-neural-lms/lecture-04-neural-lms.ipynb) contain micrograd and LSTM material. Those external experiments may require extra dependencies and downloads; they are optional and are not run by this notebook.

## E03 · One attention row by hand

**5 minutes.** These invented vectors have key dimension $d_k=2$. Focus on query 2, $q_2=(\sqrt{2},0)$.

| Position | Label | Key | Value |
| --- | --- | --- | --- |
| 1 | red | (1, 0) | (1, 0) |
| 2 | key | (0, 1) | (0, 2) |
| 3 | opens | (1, 1) | (3, 1) |

Compute $s_j=q_2^\top k_j/\sqrt{2}$, then softmax weights and $o_2=\sum_j a_jv_j$. All three keys are allowed in this first example. Use $e\approx2.7183$.

Source: [Vaswani et al. (2017), §3.2.1](https://arxiv.org/abs/1706.03762). The vectors here are hand-chosen teaching values, not trained representations.

In [ ]:
root_two = math.sqrt(2)
toy_q = torch.tensor([[[0., root_two], [root_two, 0.], [root_two, root_two]]], dtype=torch.float64)
toy_k = torch.tensor([[[1., 0.], [0., 1.], [1., 1.]]], dtype=torch.float64)
toy_v = torch.tensor([[[1., 0.], [0., 2.], [3., 1.]]], dtype=torch.float64)
toy_scores = toy_q @ toy_k.transpose(-2, -1) / math.sqrt(toy_q.shape[-1])
toy_weights = torch.softmax(toy_scores, dim=-1)
toy_outputs = toy_weights @ toy_v
print('query 2 scores:', toy_scores[0, 1].tolist())
print('query 2 weights:', toy_weights[0, 1].tolist())
print('query 2 output:', toy_outputs[0, 1].tolist())

**Check:** scores $(1,0,1)$, weights $\frac{1}{2e+1}(e,1,e)\approx(0.4223,0.1554,0.4223)$, and output $(1.6893,0.7330)$.

The weighted average lives in the **value** space. The query/key width controls score computation and need not equal the value width. Softmax normalizes **over keys for each query**, not over queries or the batch.

## E04 · Causal connections

**4 minutes.** At input position $t$, the model predicts the token at $t+1$. Which keys can it read? Sketch a 3-by-3 allowed-connection matrix. Does position 2's output change if only position 3's value changes?

Use `True` to mean **allowed**. The diagonal is allowed because the current input token is already known. Apply the mask to scores **before** softmax. Every query in this example has at least one allowed key.

In [ ]:
allowed = torch.ones(3, 3, dtype=torch.bool).tril()
causal_scores = toy_scores.masked_fill(~allowed, float('-inf'))
causal_weights = torch.softmax(causal_scores, dim=-1)
causal_outputs = causal_weights @ toy_v
changed_values = toy_v.clone()
changed_values[:, 2] += 10
causal_changed = causal_weights @ changed_values
unmasked_changed = toy_weights @ changed_values
post_softmax_mask = toy_weights * allowed
print('allowed mask:', allowed.tolist())
print('query 2 masked weights:', causal_weights[0, 1].tolist())
print('query 2 masked output:', causal_outputs[0, 1].tolist())
print('masked future change:', (causal_changed[0, 1] - causal_outputs[0, 1]).tolist())
print('unmasked future change:', (unmasked_changed[0, 1] - toy_outputs[0, 1]).tolist())
print('wrong post-softmax row sum:', post_softmax_mask[0, 1].sum().item())

**Check:** the allowed rows are `(1,0,0)`, `(1,1,0)`, `(1,1,1)`. Query 2's masked weights are $(0.7311,0.2689,0)$ and its output is $(0.7311,0.5379)$. Changing value 3 leaves that output unchanged. Without masking, both coordinates change by about $4.2232$.

Zeroing weights after softmax without renormalizing gives a query-2 row sum of about $0.5777$. Masking the scores first gives the intended normalized distribution. A row containing only negative infinity has undefined softmax; the implementation below explicitly rejects fully masked rows.

### A small attention API

Implement one head for rank-3 tensors: Q `(B, Tq, dk)`, K `(B, Tk, dk)`, and V `(B, Tk, dv)`. Self-attention uses `Tq = Tk`; the function also accepts different query/key lengths. A boolean allowed mask broadcasts to `(B, Tq, Tk)`.

The checks below make failures visible during classroom experiments. Inputs use the same floating dtype and device. There is no dropout in this example. The returned tuple contains output vectors and attention weights.

In [ ]:
def scaled_attention(q, k, v, allowed=None):
    if any(tensor.ndim != 3 for tensor in (q, k, v)):
        raise ValueError('Q, K, and V must be rank 3')
    if (q.shape[0] != k.shape[0] or k.shape[:2] != v.shape[:2]
            or q.shape[-1] != k.shape[-1] or min(*q.shape, *k.shape, *v.shape) < 1):
        raise ValueError('incompatible attention shapes')
    if (not all(tensor.is_floating_point() for tensor in (q, k, v))
            or len({tensor.dtype for tensor in (q, k, v)}) != 1
            or len({tensor.device for tensor in (q, k, v)}) != 1):
        raise ValueError('Q, K, and V need the same floating dtype and device')
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    if allowed is not None:
        if allowed.dtype != torch.bool or allowed.device != q.device:
            raise ValueError('allowed must be boolean on the input device')
        try:
            allowed = torch.broadcast_to(allowed, scores.shape)
        except RuntimeError as error:
            raise ValueError('mask must broadcast to the score shape') from error
        if not allowed.any(dim=-1).all():
            raise ValueError('each query needs at least one allowed key')
        scores = scores.masked_fill(~allowed, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return weights @ v, weights


class SingleHead(torch.nn.Module):
    def __init__(self, model_width, key_width, value_width):
        super().__init__()
        self.query = torch.nn.Linear(model_width, key_width, bias=False)
        self.key = torch.nn.Linear(model_width, key_width, bias=False)
        self.value = torch.nn.Linear(model_width, value_width, bias=False)

    def forward(self, x):
        mask = torch.ones(x.shape[1], x.shape[1], device=x.device, dtype=torch.bool).tril()
        return scaled_attention(self.query(x), self.key(x), self.value(x), mask)

## E05 · Output and gradient checks

**6 minutes.** Before running the implementation above, write the three core operations: score computation, masked softmax, and value aggregation. Explain what would fail if softmax used `dim=-2` or the mask were transposed.

Then run the checks below. Compare with a reference that loops over batches, queries, and **only the allowed keys**. Check gradients under a nonuniform scalar probe, then use finite differences. Finally, change future input vectors before the learned Q/K/V projections and verify causality end to end through this head.

This adapts the implementation-and-test approach of [CS336 Assignment 1](https://github.com/stanford-cs336/assignment1-basics) and [Lecture 2's gradient examples](https://github.com/stanford-cs336/lectures/blob/main/lecture_02.py). These small checks are original course exercises.

In [ ]:
def attention_reference(q, k, v, allowed):
    mask = torch.broadcast_to(allowed, (q.shape[0], q.shape[1], k.shape[1]))
    batches = []
    for batch in range(q.shape[0]):
        rows = []
        for query in range(q.shape[1]):
            keys = mask[batch, query].nonzero().flatten().tolist()
            scores = torch.stack([torch.dot(q[batch, query], k[batch, key])
                                  / math.sqrt(q.shape[-1]) for key in keys])
            weights = torch.softmax(scores, dim=0)
            rows.append(sum(weight * v[batch, key] for weight, key in zip(weights, keys)))
        batches.append(torch.stack(rows))
    return torch.stack(batches)


actual, actual_weights = scaled_attention(toy_q, toy_k, toy_v, allowed)
reference = attention_reference(toy_q, toy_k, toy_v, allowed)
forward_error = (actual - reference).abs().max().item()
torch.testing.assert_close(actual, reference, rtol=1e-10, atol=1e-10)
print('maximum forward error:', forward_error)

In [ ]:
torch.manual_seed(11)
inputs = tuple(torch.randn(1, 3, 2, dtype=torch.float64, requires_grad=True) for _ in range(3))
reference_inputs = tuple(tensor.detach().clone().requires_grad_() for tensor in inputs)
probe = torch.tensor([[[1., -2.], [0.5, 3.], [-1., 2.]]], dtype=torch.float64)
output, _ = scaled_attention(*inputs, allowed)
reference_output = attention_reference(*reference_inputs, allowed)
grads = torch.autograd.grad((output * probe).sum(), inputs)
reference_grads = torch.autograd.grad((reference_output * probe).sum(), reference_inputs)
gradient_errors = {}
for name, actual_grad, reference_grad in zip(['Q', 'K', 'V'], grads, reference_grads):
    torch.testing.assert_close(actual_grad, reference_grad, rtol=1e-10, atol=1e-10)
    gradient_errors[name] = (actual_grad - reference_grad).abs().max().item()
gradcheck_ok = torch.autograd.gradcheck(
    lambda q, k, v: scaled_attention(q, k, v, allowed)[0], inputs,
    eps=1e-6, atol=1e-5, rtol=1e-3,
)
print('gradient maximum errors:', gradient_errors)
print('finite-difference gradcheck:', gradcheck_ok)

In [ ]:
torch.manual_seed(19)
head = SingleHead(4, 2, 2).double()
head_inputs = torch.randn(1, 4, 4, dtype=torch.float64, requires_grad=True)
head_outputs, head_weights = head(head_inputs)
future_inputs = head_inputs.detach().clone()
future_inputs[:, 2:] += 10
changed_outputs, _ = head(future_inputs)
prefix_error = (head_outputs[:, :2] - changed_outputs[:, :2]).abs().max().item()
prefix_gradient = torch.autograd.grad(head_outputs[:, :2].square().sum(), head_inputs)[0]
future_gradient_max = prefix_gradient[:, 2:].abs().max().item()
torch.testing.assert_close(head_outputs[:, :2], changed_outputs[:, :2], rtol=0, atol=0)
assert future_gradient_max == 0
print('earlier output change:', prefix_error)
print('gradient from earlier outputs to future inputs:', future_gradient_max)

unmasked_outputs, _ = scaled_attention(head.query(head_inputs), head.key(head_inputs), head.value(head_inputs))
unmasked_future, _ = scaled_attention(head.query(future_inputs), head.key(future_inputs), head.value(future_inputs))
unmasked_prefix_change = (unmasked_outputs[:, :2] - unmasked_future[:, :2]).abs().max().item()
print('same perturbation without mask:', unmasked_prefix_change)

**Check:** forward and gradient comparisons pass within the stated float64 tolerances, and `gradcheck` returns `True`. Both the earlier-output change and the gradient to future inputs are zero for the masked head. The unmasked control changes.

For the gradient check, the scalar loss uses **only earlier outputs**. A loss over all sequence positions can legitimately have gradients to later inputs. Output agreement at one input is insufficient by itself; the separate gradient and perturbation checks test different properties.

The learned projection parameters are shared across positions. Gradient paths into those parameters remain active. The causal statement concerns dependence of earlier outputs on later **input values in this forward pass**, not information already learned in model parameters.

## P01 · Attention resource accounting

For one materialized dense score tensor $(B,T,T)$ in fp32, compute $4BT^2$ bytes. Use $B=1$ and $T\in\{512,1024,2048\}$, reporting MiB ($2^{20}$ bytes).

For $d_k=d_v=d$, the two dense products $QK^\top$ and $AV$ cost approximately $4BT^2d$ FLOPs if a multiply and an add count as two operations. This omits projections and softmax. A causal mask does not automatically reduce work in this dense implementation. Efficient kernels can avoid materializing the whole score matrix; they belong to the later systems discussion.

Apply the accounting method from [CS336 Lecture 2](https://github.com/stanford-cs336/lectures/blob/main/lecture_02.py) to the head we just implemented.

In [ ]:
score_mib = {length: 4 * length**2 / 2**20 for length in (512, 1024, 2048)}
dense_head_flops = {length: 4 * length**2 * 64 for length in score_mib}
print('one fp32 score matrix, MiB:', score_mib)
print('two dense products, FLOPs at d=64:', dense_head_flops)

## P02 · Paired key/value permutations

Keep query 3 fixed and permute its allowed key/value pairs **together**. Predict the output. Every key is allowed for this last query, so all pairs can be reordered together.

This operation leaves the weighted sum unchanged. A plain dot-product score does not explicitly represent relative distance. Causal masking supplies an ordering constraint; positional representations will add position information in Lecture 05. This example does not claim that an entire causally masked network is permutation invariant.

In [ ]:
permutation = torch.tensor([1, 0, 2])
last_query = toy_q[:, 2:3]
original_last, _ = scaled_attention(last_query, toy_k, toy_v)
permuted_last, _ = scaled_attention(last_query, toy_k[:, permutation], toy_v[:, permutation])
pair_permutation_error = (original_last - permuted_last).abs().max().item()
torch.testing.assert_close(original_last, permuted_last, rtol=1e-12, atol=1e-12)
print('paired permutation output error:', pair_permutation_error)

## Readings and continuation

- [Bengio et al. (2003), §2](https://www.jmlr.org/papers/v3/bengio03a.html): neural probabilistic language models.
- [Bahdanau et al. (ICLR 2015), §§2–3](https://arxiv.org/abs/1409.0473): the encoder–decoder context bottleneck and attention motivation.
- [Vaswani et al. (2017), §§3.2.1 and 3.2.3](https://arxiv.org/abs/1706.03762): scaled dot-product attention and masking. Section 3.5 previews positions.
- [CS336 Lecture 2](https://github.com/stanford-cs336/lectures/blob/main/lecture_02.py): named tensor axes, gradients, and resource accounting.
- [CS336 Assignment 1](https://github.com/stanford-cs336/assignment1-basics): implementation and component checks. Our notebook uses small, self-contained course exercises.

Lecture 05 on October 10 assembles the Transformer architecture. [Optional reading](https://github.com/baojian/llm-26-fall/blob/main/slides/lecture-04/optional-reading.md) maps the retained Spring material and further experiments.